# Project Nano-Myo
## Notebook 01 - Data Loading

Prepare the raw NinaPro DB5 subject archives for downstream processing.

Outputs are written to `Nano_Myo/extracted/` with one directory per subject. Only E2 and E3 `.mat` files are extracted; E1 is ignored.

## Expected Drive Layout

Raw input:

```text
MyDrive/
  Nano_Myo/
    s1.zip
    s2.zip
    ...
    s10.zip
```

Generated output:

```text
MyDrive/
  Nano_Myo/
    extracted/
      s1/
        *_E2_*.mat
        *_E3_*.mat
      ...
      s10/
        *_E2_*.mat
        *_E3_*.mat
      extraction_manifest.json
```


## Step 1 - Mount Drive and Configure Paths

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print('Not running in Colab. Drive mount skipped.')

from pathlib import Path
import json
import shutil
import zipfile
from datetime import datetime, timezone

import numpy as np
import scipy.io

NANO_MYO_ROOT = Path('/content/drive/MyDrive/Nano_Myo')
EXTRACTED_DIR = NANO_MYO_ROOT / 'extracted'

SUBJECTS = [f's{i}' for i in range(1, 11)]
TARGET_EXERCISES = ('E2', 'E3')

OVERWRITE_EXISTING = False

print(f'Project root:  {NANO_MYO_ROOT}')
print(f'Extracted dir: {EXTRACTED_DIR}')
print(f'Subjects:      {SUBJECTS}')
print(f'Exercises:     {TARGET_EXERCISES}')

## Step 2 - Validate Inputs

The pipeline stops immediately if any subject archive is missing.

In [ ]:
def subject_zip_path(subject_id):
    return NANO_MYO_ROOT / f'{subject_id}.zip'


def validate_raw_inputs():
    if not NANO_MYO_ROOT.exists():
        raise FileNotFoundError(
            f'Project folder not found: {NANO_MYO_ROOT}. '
            'Check the Drive path or folder name.'
        )

    missing = [str(subject_zip_path(s)) for s in SUBJECTS if not subject_zip_path(s).exists()]
    if missing:
        message = 'Missing subject ZIP file(s):\n' + '\n'.join(f'  - {p}' for p in missing)
        raise FileNotFoundError(message)

    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
    print('Raw input validation passed.')


validate_raw_inputs()

## Step 3 - Inspect `s1.zip`

Preview the archive structure and target files before extraction.

In [ ]:
def is_target_mat_member(member_name):
    filename = Path(member_name).name
    upper_name = filename.upper()
    return filename.lower().endswith('.mat') and any(f'_{ex}_' in upper_name for ex in TARGET_EXERCISES)


def list_zip_members(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        return sorted(zf.namelist())


s1_zip = subject_zip_path('s1')
members = list_zip_members(s1_zip)
target_members = [m for m in members if is_target_mat_member(m)]

print(f'Inspecting: {s1_zip}')
print(f'Total archive members: {len(members)}')
print(f'Target E2/E3 .mat files: {len(target_members)}')

print('\nTarget files that will be extracted from s1.zip:')
for name in target_members:
    print(f'  {name}')

print('\nArchive preview:')
for name in members[:40]:
    print(f'  {name}')
if len(members) > 40:
    print(f'  ... {len(members) - 40} more entries')

if len(target_members) == 0:
    raise RuntimeError('No E2/E3 .mat files found in s1.zip. Check filename conventions before continuing.')

## Step 4 - Extract E2 and E3 for Every Subject

Existing non-empty files are skipped unless `OVERWRITE_EXISTING = True`.

In [ ]:
def classify_exercise_from_filename(filename):
    upper_name = filename.upper()
    for exercise in TARGET_EXERCISES:
        if f'_{exercise}_' in upper_name:
            return exercise
    return None


def extract_member_flat(zf, member_name, output_dir, overwrite=False):
    filename = Path(member_name).name
    if not filename:
        raise ValueError(f'Archive member has no filename: {member_name}')

    output_path = output_dir / filename
    existing_ok = output_path.exists() and output_path.stat().st_size > 0
    if existing_ok and not overwrite:
        return output_path, 'skipped_existing'

    with zf.open(member_name, 'r') as src, output_path.open('wb') as dst:
        shutil.copyfileobj(src, dst)

    return output_path, 'extracted'


def extract_subject(subject_id, overwrite=False):
    zip_path = subject_zip_path(subject_id)
    output_dir = EXTRACTED_DIR / subject_id
    output_dir.mkdir(parents=True, exist_ok=True)

    records = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        target_members = [m for m in zf.namelist() if is_target_mat_member(m)]

        for member_name in sorted(target_members):
            output_path, status = extract_member_flat(zf, member_name, output_dir, overwrite=overwrite)
            records.append({
                'subject': subject_id,
                'exercise': classify_exercise_from_filename(output_path.name),
                'archive_member': member_name,
                'output_path': str(output_path),
                'size_bytes': output_path.stat().st_size,
                'status': status,
            })

    return records


all_records = []
for subject_id in SUBJECTS:
    records = extract_subject(subject_id, overwrite=OVERWRITE_EXISTING)
    all_records.extend(records)
    status_counts = {}
    for record in records:
        status_counts[record['status']] = status_counts.get(record['status'], 0) + 1
    print(f'{subject_id}: {len(records)} target file(s), statuses={status_counts}')

print(f'\nTotal target files processed: {len(all_records)}')

## Step 5 - Validate Extracted Layout

Each subject must have at least one E2 file and one E3 file.

In [ ]:
def extracted_mat_files(subject_id):
    subject_dir = EXTRACTED_DIR / subject_id
    if not subject_dir.exists():
        return []
    return sorted(p for p in subject_dir.iterdir() if p.is_file() and p.suffix.lower() == '.mat')


def validate_extracted_layout():
    errors = []
    summary = []

    for subject_id in SUBJECTS:
        files = extracted_mat_files(subject_id)
        by_exercise = {exercise: [] for exercise in TARGET_EXERCISES}

        for path in files:
            exercise = classify_exercise_from_filename(path.name)
            if exercise in by_exercise:
                by_exercise[exercise].append(path)

        for exercise in TARGET_EXERCISES:
            if not by_exercise[exercise]:
                errors.append(f'{subject_id} is missing {exercise}')

        summary.append({
            'subject': subject_id,
            'total_mat_files': len(files),
            'e2_files': len(by_exercise['E2']),
            'e3_files': len(by_exercise['E3']),
            'files': [p.name for p in files],
        })

    if errors:
        raise RuntimeError('Extracted layout validation failed:\n' + '\n'.join(f'  - {e}' for e in errors))

    return summary


layout_summary = validate_extracted_layout()
print('Extracted layout validation passed.\n')
for row in layout_summary:
    print(f"{row['subject']}: E2={row['e2_files']}, E3={row['e3_files']}, total_mat={row['total_mat_files']}")

## Step 6 - Write Extraction Manifest

Record extracted files, paths, sizes, and status.

In [ ]:
manifest_path = EXTRACTED_DIR / 'extraction_manifest.json'

manifest = {
    'project': 'Nano-Myo',
    'dataset': 'NinaPro DB5',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(NANO_MYO_ROOT),
    'extracted_dir': str(EXTRACTED_DIR),
    'subjects': SUBJECTS,
    'target_exercises': list(TARGET_EXERCISES),
    'overwrite_existing': OVERWRITE_EXISTING,
    'records': all_records,
    'layout_summary': layout_summary,
}

with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f'Manifest written: {manifest_path}')

## Step 7 - Sanity Check One `.mat` File

Verify the MATLAB keys and signal dimensions expected by Notebook 02.

In [ ]:
def mat_key(mat, required_name):
    wanted = required_name.lower()
    for key in mat.keys():
        if key.lower() == wanted:
            return key
    available = [k for k in mat.keys() if not k.startswith('__')]
    raise KeyError(f'Missing key {required_name!r}. Available keys: {available}')


def first_file_for(subject_id, exercise):
    matches = [p for p in extracted_mat_files(subject_id) if f'_{exercise}_' in p.name.upper()]
    if not matches:
        raise FileNotFoundError(f'No {exercise} file found for {subject_id}')
    return matches[0]


def summarize_mat_file(path):
    mat = scipy.io.loadmat(path)
    data_keys = [k for k in mat.keys() if not k.startswith('__')]

    emg = np.asarray(mat[mat_key(mat, 'emg')])
    restimulus = np.asarray(mat[mat_key(mat, 'restimulus')]).squeeze()
    rerepetition = np.asarray(mat[mat_key(mat, 'rerepetition')]).squeeze()

    if emg.ndim != 2:
        raise ValueError(f'Expected emg to be 2D, got shape {emg.shape}')
    if emg.shape[0] != restimulus.shape[0]:
        raise ValueError('emg and restimulus have different timestep counts')
    if emg.shape[0] != rerepetition.shape[0]:
        raise ValueError('emg and rerepetition have different timestep counts')
    if emg.shape[1] < 16:
        raise ValueError(f'Expected at least 16 EMG channels, got {emg.shape[1]}')

    print(f'Loaded: {path}')
    print('\nKeys:')
    for key in data_keys:
        value = mat[key]
        shape = getattr(value, 'shape', None)
        dtype = getattr(value, 'dtype', None)
        print(f'  {key}: shape={shape}, dtype={dtype}')

    print('\nEMG:')
    print(f'  shape: {emg.shape}  (timesteps x channels)')
    print(f'  dtype: {emg.dtype}')
    print(f'  range: [{emg.min():.4f}, {emg.max():.4f}]')
    print(f'  approx duration at 200 Hz: {emg.shape[0] / 200:.1f} seconds')

    print('\nrestimulus labels:')
    for label in np.unique(restimulus):
        count = int(np.sum(restimulus == label))
        tag = ' (rest)' if int(label) == 0 else ''
        print(f'  {int(label):>2}{tag}: {count:>7} timesteps, approx {count / 200:.1f} seconds')

    print('\nrerepetition values:')
    print(f'  {np.unique(rerepetition).astype(int).tolist()}')

    return {
        'path': str(path),
        'emg_shape': tuple(int(v) for v in emg.shape),
        'restimulus_labels': np.unique(restimulus).astype(int).tolist(),
        'rerepetition_values': np.unique(rerepetition).astype(int).tolist(),
    }


sanity_path = first_file_for('s1', 'E2')
sanity_summary = summarize_mat_file(sanity_path)
print('\nSanity check passed.')

## Step 8 - Final Layout Summary

In [ ]:
print('Nano_Myo/')
print('  extracted/')
for subject_id in SUBJECTS:
    print(f'    {subject_id}/')
    for path in extracted_mat_files(subject_id):
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'      {path.name}  ({size_mb:.2f} MB)')
print(f'    {manifest_path.name}')

print('\nNotebook 01 complete. Next: 02_feature_extraction.ipynb')